<a href="https://colab.research.google.com/github/AnnabelleMcSharry/AI-ML-Car-Data/blob/main/Task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install apyori
from apyori import apriori
import pandas as pd



  Preparing metadata (setup.py) ... done
  Created wheel for apyori: filename=apyori-1.1.2-py3-none-any.whl size=5954 sha256=35cc37dd8814c537f32bb9fdc63eec0aee4d1baed6155fdae900e58c4ebe4d0c
  Stored in directory: /root/.cache/pip/wheels/7f/49/e3/42c73b19a264de37129fadaa0c52f26cf50e87de08fb9804af
Successfully built apyori


In [7]:
Invoices = pd.read_excel('Online Retail.xlsx')
print('Dimensions of dataset are :', Invoices.shape)
print(Invoices.head())

Dimensions of dataset are : (541909, 8)
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  
0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2 2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
3 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
4 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  


In [8]:
pd.set_option('display.max_rows', None)
print(Invoices["Description"].value_counts())

Description
WHITE HANGING HEART T-LIGHT HOLDER     2369
REGENCY CAKESTAND 3 TIER               2200
JUMBO BAG RED RETROSPOT                2159
PARTY BUNTING                          1727
LUNCH BAG RED RETROSPOT                1638
ASSORTED COLOUR BIRD ORNAMENT          1501
SET OF 3 CAKE TINS PANTRY DESIGN       1473
PACK OF 72 RETROSPOT CAKE CASES        1385
LUNCH BAG  BLACK SKULL.                1350
NATURAL SLATE HEART CHALKBOARD         1280
POSTAGE                                1252
JUMBO BAG PINK POLKADOT                1251
HEART OF WICKER SMALL                  1237
JAM MAKING SET WITH JARS               1229
JUMBO STORAGE BAG SUKI                 1214
PAPER CHAIN KIT 50'S CHRISTMAS         1210
JUMBO SHOPPER VINTAGE RED PAISLEY      1202
LUNCH BAG CARS BLUE                    1197
LUNCH BAG SPACEBOY DESIGN              1192
JAM MAKING SET PRINTED                 1182
RECIPE BOX PANTRY YELLOW DESIGN        1180
SPOTTY BUNTING                         1172
LUNCH BAG SUKI DESIG

In [10]:
description_counts = Invoices['Description'].value_counts()
descriptions_to_keep = description_counts[description_counts >= 10].index
Invoices = Invoices[Invoices['Description'].isin(descriptions_to_keep)]
print('Shape of Invoices after filtering:', Invoices.shape)
print('First 5 rows of filtered Invoices:\n', Invoices.head())

Shape of Invoices after filtering: (537019, 8)
First 5 rows of filtered Invoices:
   InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  
0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2 2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
3 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
4 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  


In [9]:
duplicates = Invoices.duplicated()
print(f"Number of duplicate rows: {duplicates.sum()}")

# Check for missing data
missing_data = Invoices.isnull().sum()
print("\nMissing data in each column:")
print(missing_data)

Number of duplicate rows: 5268

Missing data in each column:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


In [11]:
Invoices.drop_duplicates(inplace=True)
print(f"Shape after dropping duplicates: {Invoices.shape}")

# Dropping rows where 'Description' or 'CustomerID' is NaN
Invoices.dropna(subset=['Description', 'CustomerID'], inplace=True)
print(f"Shape after dropping missing values: {Invoices.shape}")

Shape after dropping duplicates: (531768, 8)
Shape after dropping missing values: (399361, 8)


### Task 4 - Grouping the data for Market Basket Analysis

First, we'll ensure the `InvoiceDate` column is in datetime format and then extract the `YearMonth` to facilitate monthly grouping. We'll then create three different groupings:

1.  **`InvoiceMonth_group`**: Lists all item descriptions purchased within each month.
2.  **`Customer_group`**: Lists all item descriptions purchased by each unique customer.
3.  **`Customer_Month_Group`**: Lists all item descriptions purchased by each unique customer within each month.

In [13]:
import pandas as pd

# The 'InvoiceDate' column is already in datetime format from pd.read_excel.
# No need for: Invoices['InvoiceDate'] = pd.to_datetime(Invoices['InvoiceDate'])

# Extract Year-Month and create a new column
Invoices['YearMonth'] = Invoices['InvoiceDate'].dt.to_period('M')

# Grouping 1: Items purchased per month
InvoiceMonth_group = Invoices.groupby('YearMonth')['Description'].apply(list).reset_index()

# Grouping 2: Items purchased by each customer
Customer_group = Invoices.groupby('CustomerID')['Description'].apply(list).reset_index()

# Grouping 3: Items purchased by each customer per month
Customer_Month_Group = Invoices.groupby(['CustomerID','YearMonth'])['Description'].apply(list).reset_index()


print('InvoiceMonth_group head (items purchased per month):')
display(InvoiceMonth_group.head())
print('\nCustomer_group head (all items purchased by each customer):')
display(Customer_group.head())
print('\nCustomer_Month_Group head (items purchased by each customer per month):')
display(Customer_Month_Group.head())

InvoiceMonth_group head (items purchased per month):


,YearMonth,Description
0,2010-12,"[WHITE HANGING HEART T-LIGHT HOLDER, WHITE MET..."
1,2011-01,"[JUMBO BAG PINK POLKADOT, BLUE POLKADOT WRAP, ..."
2,2011-02,"[RED SPOT CERAMIC DRAWER KNOB, RED STRIPE CERA..."
3,2011-03,"[DOORMAT UNION JACK GUNS AND ROSES, DOORMAT HE..."
4,2011-04,"[LUNCH BAG DOLLY GIRL DESIGN, HEART IVORY TREL..."



Customer_group head (all items purchased by each customer):


,CustomerID,Description
0,12346.0,"[MEDIUM CERAMIC TOP STORAGE JAR, MEDIUM CERAMI..."
1,12347.0,"[BLACK CANDELABRA T-LIGHT HOLDER, AIRLINE BAG ..."
2,12348.0,"[72 SWEETHEART FAIRY CAKE CASES, 60 CAKE CASES..."
3,12349.0,"[PARISIENNE CURIO CABINET, SWEETHEART WALL TID..."
4,12350.0,"[CHOCOLATE THIS WAY METAL SIGN, METAL SIGN NEI..."



Customer_Month_Group head (items purchased by each customer per month):


,CustomerID,YearMonth,Description
0,12346.0,2011-01,"[MEDIUM CERAMIC TOP STORAGE JAR, MEDIUM CERAMI..."
1,12347.0,2010-12,"[BLACK CANDELABRA T-LIGHT HOLDER, AIRLINE BAG ..."
2,12347.0,2011-01,"[PINK NEW BAROQUECANDLESTICK CANDLE, BLUE NEW ..."
3,12347.0,2011-04,"[AIRLINE BAG VINTAGE JET SET WHITE, AIRLINE BA..."
4,12347.0,2011-06,"[RABBIT NIGHT LIGHT, REGENCY TEA STRAINER, REG..."
